# Cat and Dog Classifier

This notebook loads the CSV labels and images, trains a CNN, evaluates it, and saves the model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
RANDOM_STATE = 42

# Run the notebook with CNN/Cat-Dog-Classifier as the working directory.
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / 'cat_dog.csv'
IMAGE_DIR = BASE_DIR / 'cat_dog'
MODEL_PATH = BASE_DIR / 'cat_dog_classifier.keras'


In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['labels'],
)
print(f'Training samples: {len(train_df)}')
print(f'Testing samples: {len(test_df)}')

In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def make_dataset(frame, shuffle=False):
    # Convert paths to Python strings before TensorFlow builds the dataset.
    paths = [str(IMAGE_DIR / filename) for filename in frame['image']]
    labels = frame['labels'].astype('int32').to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        dataset = dataset.shuffle(len(frame), seed=RANDOM_STATE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_df, shuffle=True)
test_dataset = make_dataset(test_df)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(2, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            MODEL_PATH, monitor='val_accuracy', save_best_only=True
        ),
    ],
)

In [ ]:
loss, accuracy = model.evaluate(test_dataset, verbose=0)
print(f'Test accuracy: {accuracy:.4f}')
print(f'Saved model: {MODEL_PATH}')

In [ ]:
def predict_image(image_path):
    image = tf.io.read_file(str(image_path))
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    probabilities = model.predict(tf.expand_dims(image, 0), verbose=0)[0]
    class_index = int(np.argmax(probabilities))
    class_name = 'dog' if class_index == 1 else 'cat'
    return class_name, float(probabilities[class_index])

# Example:
# predict_image(IMAGE_DIR / 'cat.0.jpg')